# Train UnlimitedPipe's `ask` model (Qwen2.5 0.5B + LoRA)

Fine-tunes a small open model to answer from numbered sources the way `unlimited ask` needs:
cite the right source, invent nothing, say plainly when the sources do not cover the question,
in English and Thai. Data: the private dataset `unlimitedpipe/ask-sft`.

**Before running**
1. Settings → Accelerator: **GPU T4 x2** (or P100). Settings → Internet: **on** (needs a
   phone-verified Kaggle account).
2. Add-ons → Secrets → add **`HF_TOKEN`**: a Hugging Face fine-grained token with write access
   to the `unlimitedpipe` organization only (create a separate one for Kaggle).
3. Run All. About an hour on a T4. It uploads, privately, `unlimitedpipe/ask-0.5b` (the model)
   and `unlimitedpipe/ask-0.5b-GGUF` (the file Ollama runs).

In [ ]:
!pip install -q -U "trl>=0.20" "peft>=0.13" "transformers>=4.46" datasets accelerate

In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REPO, GGUF_REPO = "unlimitedpipe/ask-0.5b", "unlimitedpipe/ask-0.5b-GGUF"
EPOCHS = 1

In [ ]:
from datasets import load_dataset

data = load_dataset(
    "unlimitedpipe/ask-sft", data_files={"train": "train.jsonl", "test": "test.jsonl"}
)
# Prompt and completion: the loss is on the answer only, not on the sources.
data = data.map(
    lambda e: {"prompt": e["messages"][:1], "completion": e["messages"][1:]},
    remove_columns=data["train"].column_names,
)
print(data)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=BASE,
    train_dataset=data["train"],
    eval_dataset=data["test"],
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"
    ),
    args=SFTConfig(
        output_dir="ask-lora",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=20,
        max_length=1536,
        fp16=True,
        logging_steps=25,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
    ),
)
trainer.train()
print(trainer.evaluate())

In [ ]:
# A quick look: two test questions, answered by the trained model.
import torch

tok = trainer.processing_class
model = trainer.model
model.eval()
for example in data["test"].select(range(2)):
    ids = tok.apply_chat_template(
        example["prompt"], add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=120, do_sample=False)
    print(example["prompt"][0]["content"].split("Question:")[-1].strip())
    print("->", tok.decode(out[0][ids.shape[1] :], skip_special_tokens=True))
    print("expected:", example["completion"][0]["content"], "\n")

In [ ]:
# Merge the LoRA weights into the model and upload it (private).
merged = trainer.model.merge_and_unload()
merged.save_pretrained("ask-merged", safe_serialization=True)
tok.save_pretrained("ask-merged")
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path="ask-merged",
    repo_id=MODEL_REPO,
    commit_message=f"LoRA r16, {EPOCHS} epoch(s) on ask-sft",
)

In [ ]:
# The GGUF file Ollama runs (8-bit, about 530 MB), uploaded privately.
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q ./llama.cpp/gguf-py sentencepiece
!python llama.cpp/convert_hf_to_gguf.py ask-merged --outtype q8_0 --outfile ask-0.5b-q8_0.gguf
api.create_repo(GGUF_REPO, private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj="ask-0.5b-q8_0.gguf", path_in_repo="ask-0.5b-q8_0.gguf", repo_id=GGUF_REPO
)
print("done: tell Claude 'model trained'")